# Demo 12c — A CNN on the digits

**Class 12 · C3–C4** (slides: `slides/12_neural_networks.md`)

* **C3:** an MLP and a CNN with the same data, split and optimiser: parameters, accuracy, three seeds; the learned filters.
* **C4:** a global average pool instead of the dense head; test on shifted digits (the failure case of Demo 12a), and the same two models trained with shifted copies (data augmentation).

In [ ]:
import os

os.environ.setdefault("KERAS_BACKEND", "jax")
import time

import matplotlib.pyplot as plt
import numpy as np
import polars as pl
from sklearn.model_selection import train_test_split

from faa import demo, neural, plotlib
from faa.demo import show, slide

demo.setup("12")
BLUE, ORANGE, GREEN, RED, SKY, PURPLE, GREY = (
    plotlib.BLUE,
    plotlib.ORANGE,
    plotlib.GREEN,
    plotlib.RED,
    plotlib.SKY,
    plotlib.PURPLE,
    plotlib.GREY,
)
SEED = 0
X, y = neural.load_digits_images()
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.4, random_state=SEED, stratify=y)
Xva, Xte, yva, yte = train_test_split(Xte, yte, test_size=0.5, random_state=SEED, stratify=yte)
MAKE = {
    "MLP 64-32": lambda s: neural.mlp_model((8, 8, 1), hidden=(64, 32), seed=s),
    "CNN 8-16": lambda s: neural.cnn_model(filters=(8, 16), seed=s),
    "CNN 16-32 + global pool": lambda s: neural.cnn_model(filters=(16, 32), seed=s, gap=True),
}


def run(make, Xa, ya, tests, seeds=range(3), epochs=30):
    """Train on (Xa, ya) for each seed; return the accuracy on every test set, the models and the seconds of one fit."""
    acc = {k: [] for k in tests}
    models, secs = [], 0.0
    for s in seeds:
        m = make(s)
        t0 = time.time()
        m.fit(Xa, ya, epochs=epochs, batch_size=32, validation_data=(Xva, yva), verbose=0)
        secs = time.time() - t0
        for k, (Xt, yt) in tests.items():
            acc[k].append(m.evaluate(Xt, yt, verbose=0)[1])
        models.append(m)
    return acc, models, secs

## Slide: MLP against CNN (C3)

In [ ]:
tests = {"test": (Xte, yte)}
rows, trained = [], {}
for name, make in MAKE.items():
    acc, models, secs = run(make, Xtr, ytr, tests)
    trained[name] = models
    rows.append(
        {
            "model": name,
            "parameters": neural.n_params(models[0]),
            "test accuracy": np.mean(acc["test"]),
            "std": np.std(acc["test"]),
            "seconds per fit": secs,
        }
    )
slide("MLP against CNN on the digits")
show(pl.DataFrame(rows))
f, ax = plotlib.fig("col", 1.5)
names = [r["model"].replace(" + global pool", "\n+ pool") for r in rows]
ax.bar(names, [r["test accuracy"] for r in rows], yerr=[r["std"] for r in rows], color=[BLUE, ORANGE, GREEN])
for i, r in enumerate(rows):
    ax.text(i, 0.93, f"{r['parameters']} w", ha="center", fontsize=6, color="white")
ax.set_ylim(0.9, 1.0)
ax.set_ylabel("test accuracy")
plotlib.save(f, "c3_accuracy")

## Slide: What the first layer learned (C3)

The eight 3x3 kernels of the first convolution of the first CNN. Most look like edge or blob detectors.

In [ ]:
cnn0 = trained["CNN 8-16"][0]
W = cnn0.layers[0].get_weights()[0][:, :, 0, :]
f, ax = plt.subplots(1, 8, figsize=(plotlib.FULL, 0.7))
for k, a in enumerate(ax):
    a.imshow(W[:, :, k], cmap="RdBu_r", vmin=-np.abs(W).max(), vmax=np.abs(W).max())
    a.axis("off")
plotlib.save(f, "c3_filters")
slide("Learned filters")
print("kernel range:", W.min().round(2), W.max().round(2))
print("layer shapes:", [(ly.name, ly.output.shape[1:]) for ly in cnn0.layers])

## Slide: Shifted test digits (C4)

Both models trained on centred digits; test sets shifted by up to $s$ pixels.

In [ ]:
shift_tests = {f"s={s}": (neural.random_shift(Xte, s, seed=1), yte) for s in (0, 1, 2)}
rows = []
for name, make in MAKE.items():
    acc, _, _ = run(make, Xtr, ytr, shift_tests)
    rows.append({"model": name, **{k: np.mean(v) for k, v in acc.items()}})
slide("Shifted test digits")
show(pl.DataFrame(rows))
f, ax = plotlib.fig("col", 1.6)
for r, c in zip(rows, (BLUE, ORANGE, GREEN)):
    ax.plot([0, 1, 2], [r["s=0"], r["s=1"], r["s=2"]], "o-", color=c, label=r["model"])
ax.set_xticks([0, 1, 2])
ax.set_xlabel("maximum shift $s$ (pixels)")
ax.set_ylabel("test accuracy")
ax.legend(frameon=False)
plotlib.save(f, "c4_shift")

## Slide: Augmentation: show the shifts during training (C4)

Three shifted copies of every training digit (shift up to 2). The CNN gains most; the MLP can only memorise the shifts.

In [ ]:
Xaug = np.concatenate([Xtr] + [neural.random_shift(Xtr, 2, seed=10 + i) for i in range(3)])
yaug = np.concatenate([ytr] * 4)
rows = []
for name, make in MAKE.items():
    acc, _, _ = run(make, Xaug, yaug, shift_tests, epochs=20)
    rows.append({"model": name, **{k: np.mean(v) for k, v in acc.items()}})
slide("Augmentation with shifts")
show(pl.DataFrame(rows))

## Sweep and failure case: depth, pooling, width

What the CNN needs: with the pooling removed, the dense head grows; with one filter, the CNN is still better than chance on shifts.

In [ ]:
rows = []
for name, kw in {
    "1 filter": dict(filters=(1,)),
    "4 filters": dict(filters=(4,)),
    "8-16, no pooling": dict(filters=(8, 16), pool=False),
    "8-16": dict(filters=(8, 16)),
    "16-32": dict(filters=(16, 32)),
}.items():
    acc, models, _ = run(
        lambda s, kw=kw: neural.cnn_model(seed=s, **kw),
        Xtr,
        ytr,
        {"s=0": shift_tests["s=0"], "s=1": shift_tests["s=1"]},
        epochs=20,
    )
    rows.append(
        {"CNN": name, "parameters": neural.n_params(models[0]), "s=0": np.mean(acc["s=0"]), "s=1": np.mean(acc["s=1"])}
    )
slide("CNN variants")
show(pl.DataFrame(rows))
f, ax = plt.subplots(figsize=(1.75, 1.5))
for r in rows:
    ax.scatter(r["parameters"], r["s=1"], color=BLUE, s=14)
    ax.annotate(
        r["CNN"].replace("8-16, no pooling", "no pool"),
        (r["parameters"], r["s=1"]),
        fontsize=5.5,
        xytext=(3, 2),
        textcoords="offset points",
    )
ax.set_xscale("log")
ax.set_xlabel("weights")
ax.set_ylabel("accuracy, shift 1")
plotlib.save(f, "c4_variants")